# Student–Chatbot Interaction Log Analyzer
**Project:** *When Teachers Program the AI* — RCT on AI Tutoring for Roma Students  
**Author:** Aavash Kuikel  
**Purpose:** Demonstrate automated extraction of engagement, effort, language complexity,
help-seeking behaviour, sentiment arc, and identity context from JSON conversation logs —
the exact dimensions specified in the Terms of Reference (§3, Student interaction logs).

This notebook works on **synthetic** JSON logs that mirror the structure of real student–chatbot
sessions recorded by the PlayLab platform. The same code runs unchanged on real RCT data
once the `LOG_PATH` variable is pointed at the production export.

---
## 0. Imports & Configuration

In [ ]:
import json
import re
import math
import statistics
import collections
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

LOG_PATH = Path('../data/logs/conversation_logs.json')
print('Libraries loaded ✅')

---
## 1. Synthetic Data — Realistic Student–Chatbot Sessions

In [ ]:
SYNTHETIC_LOGS = [
    {
        "session_id": "s001", "student_id": "stu_042", "chatbot_id": "bot_math_01",
        "week": 3, "subject": "matemáticas",
        "turns": [
            {"role": "assistant", "text": "Hola! Estoy aquí para ayudarte con matemáticas. ¿En qué tema trabajamos hoy?", "timestamp_s": 0},
            {"role": "student",   "text": "no entiendo las fracciones", "timestamp_s": 12},
            {"role": "assistant", "text": "¡No te preocupes! Las fracciones pueden ser complicadas al principio. Imagina una pizza cortada en 4 partes iguales — si te comes 1 parte, tienes 1/4. ¿Lo visualizas?", "timestamp_s": 15},
            {"role": "student",   "text": "si creo que si", "timestamp_s": 45},
            {"role": "assistant", "text": "Perfecto! Ahora intentemos: si tienes 3/8 de una barra de chocolate, ¿cuántos trozos de 8 tienes?", "timestamp_s": 48},
            {"role": "student",   "text": "3 trozos?", "timestamp_s": 78},
            {"role": "assistant", "text": "¡Exactamente! Lo has entendido muy bien. Sigamos con una fracción más difícil.", "timestamp_s": 82},
            {"role": "student",   "text": "ok pero como se suman? no lo entiendo", "timestamp_s": 110},
            {"role": "assistant", "text": "Buena pregunta! Para sumar fracciones con el mismo denominador, solo sumas los numeradores. Por ejemplo: 1/4 + 2/4 = 3/4. ¿Quieres practicar uno?", "timestamp_s": 114},
            {"role": "student",   "text": "si quiero practicar, ponme un ejercicio", "timestamp_s": 150},
            {"role": "assistant", "text": "Aquí va: 2/5 + 1/5 = ?", "timestamp_s": 153},
            {"role": "student",   "text": "3/5", "timestamp_s": 175},
            {"role": "assistant", "text": "¡Perfecto! Excelente trabajo. Ves que ya lo dominas.", "timestamp_s": 178}
        ]
    },
    {
        "session_id": "s002", "student_id": "stu_017", "chatbot_id": "bot_read_02",
        "week": 3, "subject": "lectura",
        "turns": [
            {"role": "assistant", "text": "Bienvenida. Hoy vamos a trabajar la comprensión lectora. ¿Has leído el texto asignado?", "timestamp_s": 0},
            {"role": "student",   "text": "no", "timestamp_s": 30},
            {"role": "assistant", "text": "No pasa nada. ¿Quieres que te haga un resumen rápido o prefieres leerlo ahora?", "timestamp_s": 34},
            {"role": "student",   "text": "resumen", "timestamp_s": 60},
            {"role": "assistant", "text": "El texto habla de un niño gitano que aprende a leer con ayuda de su abuela en un pueblo de Andalucía.", "timestamp_s": 63},
            {"role": "student",   "text": "mi abuela también me enseñó cosas", "timestamp_s": 95},
            {"role": "assistant", "text": "Qué bonito! Las abuelas guardan mucha sabiduría. ¿Qué te enseñó la tuya?", "timestamp_s": 99},
            {"role": "student",   "text": "a hacer pan y canciones en romani", "timestamp_s": 135},
            {"role": "assistant", "text": "Eso es parte de tu identidad y es muy valioso. ¿Cuál crees que es el mensaje principal de la historia?", "timestamp_s": 140},
            {"role": "student",   "text": "que la cultura gitana es importante y que estudiar no significa olvidarla", "timestamp_s": 200},
            {"role": "assistant", "text": "¡Exacto! Una reflexión muy madura.", "timestamp_s": 205}
        ]
    },
    {
        "session_id": "s003", "student_id": "stu_099", "chatbot_id": "bot_math_01",
        "week": 4, "subject": "matemáticas",
        "turns": [
            {"role": "assistant", "text": "Hola de nuevo! ¿Cómo vas con las fracciones?", "timestamp_s": 0},
            {"role": "student",   "text": "mal, sigo sin entender", "timestamp_s": 8},
            {"role": "assistant", "text": "Está bien que lo digas. ¿Qué parte específicamente no entiendes?", "timestamp_s": 12},
            {"role": "student",   "text": "cuando los denominadores son distintos", "timestamp_s": 40},
            {"role": "assistant", "text": "Para sumar 1/2 + 1/3, necesitas el mínimo común múltiplo de 2 y 3, que es 6. Conviertes: 3/6 + 2/6 = 5/6.", "timestamp_s": 45},
            {"role": "student",   "text": "no entiendo el mcm", "timestamp_s": 90},
            {"role": "assistant", "text": "Múltiplos de 2: 2,4,6... Múltiplos de 3: 3,6,9... El primero en común es 6.", "timestamp_s": 95},
            {"role": "student",   "text": "creo que si, puedes darme otro ejemplo?", "timestamp_s": 150},
            {"role": "assistant", "text": "MCM de 4 y 6 es 12. Entonces 1/4 + 1/6 = 3/12 + 2/12 = 5/12.", "timestamp_s": 155},
            {"role": "student",   "text": "ah ya! tiene sentido", "timestamp_s": 210},
            {"role": "assistant", "text": "¿Quieres intentar uno tú solo?", "timestamp_s": 214},
            {"role": "student",   "text": "si", "timestamp_s": 230},
            {"role": "assistant", "text": "Calcula: 1/3 + 1/4", "timestamp_s": 232},
            {"role": "student",   "text": "mcm es 12, entonces 4/12 + 3/12 = 7/12", "timestamp_s": 310},
            {"role": "assistant", "text": "¡Correcto! Lo has conseguido.", "timestamp_s": 314}
        ]
    },
    {
        "session_id": "s004", "student_id": "stu_055", "chatbot_id": "bot_sci_03",
        "week": 5, "subject": "ciencias",
        "turns": [
            {"role": "assistant", "text": "Hola! Hoy estudiamos el sistema solar. ¿Cuántos planetas hay?", "timestamp_s": 0},
            {"role": "student",   "text": "8", "timestamp_s": 20},
            {"role": "assistant", "text": "¡Correcto! ¿Cuál es el más grande?", "timestamp_s": 24},
            {"role": "student",   "text": "jupiter creo", "timestamp_s": 45},
            {"role": "assistant", "text": "¡Exacto! ¿Sabes por qué Plutón ya no se considera planeta?", "timestamp_s": 49},
            {"role": "student",   "text": "no lo se", "timestamp_s": 75},
            {"role": "assistant", "text": "En 2006 la UAI redefinió planeta. Plutón no ha despejado su órbita, por eso es un planeta enano.", "timestamp_s": 79},
            {"role": "student",   "text": "que significa despejar la orbita", "timestamp_s": 120},
            {"role": "assistant", "text": "Significa que la gravedad del planeta ha absorbido o empujado otros objetos de su camino. Plutón comparte zona con muchos cuerpos en el cinturón de Kuiper.", "timestamp_s": 125},
            {"role": "student",   "text": "ah que interesante", "timestamp_s": 160}
        ]
    }
]

if LOG_PATH.exists():
    with open(LOG_PATH) as f:
        logs = json.load(f)
    print(f'Loaded {len(logs)} real sessions from {LOG_PATH}')
else:
    logs = SYNTHETIC_LOGS
    print(f'⚠️  Real log file not found. Using {len(logs)} synthetic sessions.')
    print(f'   Point LOG_PATH at the PlayLab export to run on real RCT data.')

---
## 2. Feature Extraction Pipeline

| TOR Dimension | Features Extracted |
|---|---|
| **Engagement** | turn count, session duration, response latency |
| **Effort & Persistence** | student word count, elaboration rate, follow-up ratio |
| **Language Complexity** | avg words/turn, lexical diversity (TTR), question depth |
| **Help-Seeking Behaviour** | explicit help requests, confusion signals, question count |
| **Sentiment & Tone** | lexicon-based polarity arc across turns |
| **Identity Context** | cultural/identity keyword matches |

In [ ]:
POSITIVE_WORDS = {
    'bien', 'genial', 'perfecto', 'excelente', 'gracias', 'interesante',
    'entiendo', 'entendido', 'claro', 'bueno', 'correcto', 'conseguido'
}
NEGATIVE_WORDS = {
    'no', 'mal', 'difícil', 'confundido', 'perdido', 'imposible', 'error'
}
HELP_SIGNALS = [
    r'no entiendo', r'no sé', r'puedes explicar', r'cómo se', r'qué significa',
    r'ayuda', r'no lo veo', r'no me sale', r'no comprendo', r'no lo entiendo'
]
IDENTITY_KEYWORDS = [
    r'gitano', r'gitana', r'romani', r'romá', r'caló', r'flamenco',
    r'comunidad', r'familia', r'abuela', r'abuelo', r'tradición', r'cultura'
]
QUESTION_PATTERN = re.compile(r'\?|qué|cómo|por qué|cuándo|dónde|cuál|quién', re.IGNORECASE)


def simple_sentiment(text):
    words = re.findall(r'\w+', text.lower())
    score = sum(1 for w in words if w in POSITIVE_WORDS)
    score -= sum(1 for w in words if w in NEGATIVE_WORDS)
    return max(-1.0, min(1.0, score / max(len(words), 1) * 5))


def lexical_diversity(texts):
    all_words = [w.lower() for t in texts for w in re.findall(r'\w+', t)]
    if not all_words:
        return 0.0
    return len(set(all_words)) / len(all_words)


def extract_features(session):
    turns         = session['turns']
    student_turns = [t for t in turns if t['role'] == 'student']
    n_turns          = len(turns)
    n_student_turns  = len(student_turns)
    session_duration = turns[-1]['timestamp_s'] - turns[0]['timestamp_s'] if len(turns) > 1 else 0
    latencies = [
        turns[i]['timestamp_s'] - turns[i-1]['timestamp_s']
        for i in range(1, len(turns)) if turns[i]['role'] == 'student'
    ]
    avg_latency = statistics.mean(latencies) if latencies else 0
    student_texts    = [t['text'] for t in student_turns]
    student_wc       = [len(re.findall(r'\w+', t)) for t in student_texts]
    avg_words_student = statistics.mean(student_wc) if student_wc else 0
    elaboration_rate  = sum(1 for wc in student_wc if wc > 5) / max(n_student_turns, 1)
    follow_up = sum(
        1 for i, t in enumerate(turns)
        if t['role'] == 'student' and i > 0 and QUESTION_PATTERN.search(turns[i-1]['text'])
    )
    follow_up_ratio   = follow_up / max(n_student_turns, 1)
    ttr               = lexical_diversity(student_texts)
    student_questions = sum(1 for t in student_texts if QUESTION_PATTERN.search(t))
    question_rate     = student_questions / max(n_student_turns, 1)
    help_count        = sum(
        1 for t in student_texts
        if any(re.search(p, t, re.IGNORECASE) for p in HELP_SIGNALS)
    )
    help_seeking_rate = help_count / max(n_student_turns, 1)
    sentiment_scores  = [simple_sentiment(t) for t in student_texts]
    avg_sentiment     = statistics.mean(sentiment_scores) if sentiment_scores else 0
    if len(sentiment_scores) > 1:
        n = len(sentiment_scores)
        xs = list(range(n))
        x_mean = statistics.mean(xs)
        y_mean = statistics.mean(sentiment_scores)
        slope = sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, sentiment_scores)) / \
                sum((x - x_mean) ** 2 for x in xs)
    else:
        slope = 0.0
    sentiment_arc     = 'improving' if slope > 0.02 else 'declining' if slope < -0.02 else 'stable'
    all_text          = ' '.join(student_texts)
    identity_matches  = [kw for kw in IDENTITY_KEYWORDS if re.search(kw, all_text, re.IGNORECASE)]
    identity_present  = len(identity_matches) > 0
    return {
        'session_id': session['session_id'],
        'student_id': session['student_id'],
        'chatbot_id': session['chatbot_id'],
        'week': session['week'],
        'subject': session['subject'],
        'n_turns': n_turns,
        'n_student_turns': n_student_turns,
        'session_duration_s': session_duration,
        'avg_response_latency_s': round(avg_latency, 1),
        'avg_words_per_student_turn': round(avg_words_student, 1),
        'elaboration_rate': round(elaboration_rate, 2),
        'follow_up_ratio': round(follow_up_ratio, 2),
        'lexical_diversity_ttr': round(ttr, 3),
        'question_rate': round(question_rate, 2),
        'help_seeking_count': help_count,
        'help_seeking_rate': round(help_seeking_rate, 2),
        'avg_sentiment': round(avg_sentiment, 3),
        'sentiment_arc': sentiment_arc,
        'sentiment_scores': [round(s, 2) for s in sentiment_scores],
        'identity_keywords_present': identity_present,
        'identity_keywords_matched': identity_matches,
    }


records = [extract_features(s) for s in logs]
df = pd.DataFrame(records)
print(f'Extracted features for {len(df)} sessions')
display(df[[
    'session_id', 'subject', 'n_turns', 'session_duration_s',
    'avg_words_per_student_turn', 'help_seeking_count',
    'avg_sentiment', 'sentiment_arc', 'identity_keywords_present'
]])

---
## 3. Visualization 1 — Engagement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = sns.color_palette('husl', len(df))
axes[0].bar(df['session_id'], df['n_turns'], color=colors, edgecolor='white')
for bar, val in zip(axes[0].patches, df['n_turns']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Total Turns per Session', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Session ID')
axes[0].set_ylabel('Number of Turns')
axes[1].bar(df['session_id'], df['session_duration_s'], color=colors, edgecolor='white')
for bar, val in zip(axes[1].patches, df['session_duration_s']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val}s', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_title('Session Duration (seconds)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Session ID')
axes[1].set_ylabel('Duration (s)')
fig.suptitle('Engagement Metrics by Session', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation:** Turn count and session duration are primary engagement proxies. Longer sessions with more turns suggest the student persisted rather than disengaging. In the RCT, sessions in treatment classrooms are expected to show higher average turn counts compared to control.

---
## 4. Visualization 2 — Sentiment Arc

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
palette = sns.color_palette('husl', len(df))
for i, row in df.iterrows():
    scores = row['sentiment_scores']
    x = list(range(len(scores)))
    ax.plot(x, scores, marker='o', linewidth=2,
            label=f"{row['session_id']} ({row['subject']}) — {row['sentiment_arc']}",
            color=palette[i], alpha=0.85)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6, label='Neutral baseline')
ax.set_xlabel('Student Turn Index', fontsize=12)
ax.set_ylabel('Sentiment Score', fontsize=12)
ax.set_title('Student Sentiment Arc Across Conversation Turns', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

**Interpretation:** An improving arc — sentiment trending positive — signals productive learning and successful scaffolding. In the RCT, sentiment arc would be linked to teacher-prompt tone and scaffolding depth to test whether chatbot design predicts student experience.

---
## 5. Visualization 3 — Help-Seeking & Effort

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=df, x='subject', y='help_seeking_rate', palette='Blues_d', ax=axes[0], edgecolor='white')
axes[0].set_title('Help-Seeking Rate by Subject', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Subject')
axes[0].set_ylabel('Fraction of Turns with Help Signals')
axes[0].tick_params(axis='x', rotation=15)
sns.barplot(data=df, x='subject', y='avg_words_per_student_turn', palette='Greens_d', ax=axes[1], edgecolor='white')
axes[1].set_title('Avg Words per Student Turn (Effort)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Subject')
axes[1].set_ylabel('Average Words per Turn')
axes[1].tick_params(axis='x', rotation=15)
fig.suptitle('Help-Seeking Behaviour & Effort by Subject', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation:** Help-seeking is a positive metacognitive signal. High help-seeking with low elaboration may indicate confusion without meaning-making — actionable for teachers.

---
## 6. Visualization 4 — Identity Context

In [ ]:
print('=== Identity Context Summary ===')
print(f"Sessions with cultural references: {df['identity_keywords_present'].sum()} / {len(df)}")
for _, row in df[df['identity_keywords_present']].iterrows():
    print(f"  {row['session_id']} ({row['subject']}): {row['identity_keywords_matched']}")

fig, ax = plt.subplots(figsize=(8, 4))
counts = df['identity_keywords_present'].value_counts()
labels = ['Identity context present' if v else 'Identity context absent' for v in counts.index]
ax.bar(labels, counts.values, color=['#e74c3c', '#bdc3c7'], edgecolor='white', width=0.5)
for bar, val in zip(ax.patches, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(val), ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_title('Student-Initiated Cultural/Identity References', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Sessions')
plt.tight_layout()
plt.show()

**Interpretation:** When students voluntarily introduce Roma/Gitano cultural references, it signals the chatbot has created psychological safety for cultural self-expression — a key mechanism in the RCT's identity and engagement pathway.

---
## 7. Scaling to Real RCT Data

```python
# Point at real PlayLab export
LOG_PATH = Path('/rct_data/exports/week_{week}/conversation_logs.json')

# Identical extraction code — scales to 10,000+ sessions
records = [extract_features(s) for s in logs]
df = pd.DataFrame(records)
df.to_csv(f'interaction_features_week_{week}.csv', index=False)

# Merge with outcome data for multilevel models
outcomes = pd.read_csv('student_outcomes.csv')
merged = outcomes.merge(df, on='student_id')
```

| Chatbot Design Predictor | Expected Student Interaction Outcome | Hypothesis |
|---|---|---|
| High scaffolding depth | Lower help_seeking_rate | Scaffolding reduces confusion |
| Warm/motivating tone | Improving sentiment arc | Tone shapes student affect |
| Cultural references in prompt | identity_keywords_present = True | Representation invites self-expression |
| Multiple motivational strategies | Higher elaboration_rate | Motivation drives effort |

In [ ]:
out_path = Path('../data/processed/interaction_features.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df.drop(columns=['sentiment_scores', 'identity_keywords_matched']).to_csv(out_path, index=False)
print(f'Features saved to {out_path}')
display(df.describe(include='all').T)